In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import textstat
from tqdm import tqdm

# integrating tqdm with pandas (to see a progress bar)
tqdm.pandas()

# seeing every columns
pd.set_option('display.max_columns', None)

In [2]:
df_pipeline = pd.read_csv("billboard_lyrics_features_final.csv")

df_full = pd.read_csv("spotify_millsongdata.csv")
df_full['original_index'] = df_full.index
    
df = df_full[df_full['original_index'].isin(df_pipeline['original_index'])].copy()

print(f"Filtered: {len(df)} songs")
print(df.head())
print(df.info())

Filtered: 6813 songs
        artist                 song  \
145  Aerosmith        All Your Love   
146  Aerosmith  Bacon Biscuit Blues   
148  Aerosmith        Draw The Line   
149  Aerosmith         Eat The Rich   
150  Aerosmith      Falling In Love   

                                               link  \
145        /a/aerosmith/all+your+love_20004378.html   
146  /a/aerosmith/bacon+biscuit+blues_20004481.html   
148        /a/aerosmith/draw+the+line_20004388.html   
149         /a/aerosmith/eat+the+rich_20004281.html   
150      /a/aerosmith/falling+in+love_10003058.html   

                                                  text  original_index  
145  All your love I miss lovin'  \r\nAll your kiss...             145  
146  Put your biscuits in the oven  \r\nHoney, put ...             146  
148  Checkmate honey, beat ya at your own damn game...             148  
149  Well I woke up this morning  \r\nOn the wrong ...             149  
150  You're so bad, you're so bad, you're so  \r

In [3]:
print("1. Step: Sentiment Analysis (Sentiment Analysis)")
# We are downloading and preparing the VADER emotion analysis tool of the NLTK library
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

# We take the 'compound' score of the lyrics (between -1 and Dec 1)
df['sentiment_score'] = df['text'].progress_apply(lambda x: sia.polarity_scores(str(x))['compound'])

1. Step: Sentiment Analysis (Sentiment Analysis)


100%|██████████| 6813/6813 [00:15<00:00, 435.46it/s]


In [4]:
df.head(20)

,artist,song,link,text,original_index,sentiment_score
145,Aerosmith,All Your Love,/a/aerosmith/all+your+love_20004378.html,All your love I miss lovin' \r\nAll your kiss...,145,0.9897
146,Aerosmith,Bacon Biscuit Blues,/a/aerosmith/bacon+biscuit+blues_20004481.html,"Put your biscuits in the oven \r\nHoney, put ...",146,-0.8555
148,Aerosmith,Draw The Line,/a/aerosmith/draw+the+line_20004388.html,"Checkmate honey, beat ya at your own damn game...",148,-0.8074
149,Aerosmith,Eat The Rich,/a/aerosmith/eat+the+rich_20004281.html,Well I woke up this morning \r\nOn the wrong ...,149,0.9987
150,Aerosmith,Falling In Love,/a/aerosmith/falling+in+love_10003058.html,"You're so bad, you're so bad, you're so \r\nY...",150,0.9327
151,Aerosmith,Fever,/a/aerosmith/fever_20004282.html,I got a rip in my shoes \r\nAnd a hole in my ...,151,0.9790
152,Aerosmith,Get It Up,/a/aerosmith/get+it+up_20004176.html,Take me on your rocking horse \r\nHit the lig...,152,0.9830
153,Aerosmith,I Ain't Got You,/a/aerosmith/i+aint+got+you_20004179.html,For all night Dwight \r\n \r\nGot a '56 Cadi...,153,-0.5423
154,Aerosmith,I'm Ready,/a/aerosmith/im+ready_10002994.html,Well I'm ready. \r\nAs ready as anybody can b...,154,0.9984
155,Aerosmith,Janie's Got A Gun,/a/aerosmith/janies+got+a+gun_20004444.html,"Dum, dum, dum, honey what have you done \r\nD...",155,-0.9253


In [5]:
print("\n2. Step: Cognitive Analysis (Flesch-Kincaid Readability)")
# we calculate the Flesch-Kincaid convenience score using the textstat library
# Note: This metric measures linguistic complexity based on word and sentence lengths.
df['readability_score'] = df['text'].progress_apply(lambda x: textstat.flesch_reading_ease(str(x)))


2. Step: Cognitive Analysis (Flesch-Kincaid Readability)


100%|██████████| 6813/6813 [00:04<00:00, 1527.24it/s]


In [6]:
df.head()

,artist,song,link,text,original_index,sentiment_score,readability_score
145,Aerosmith,All Your Love,/a/aerosmith/all+your+love_20004378.html,All your love I miss lovin' \r\nAll your kiss...,145,0.9897,51.873355
146,Aerosmith,Bacon Biscuit Blues,/a/aerosmith/bacon+biscuit+blues_20004481.html,"Put your biscuits in the oven \r\nHoney, put ...",146,-0.8555,-99.192081
148,Aerosmith,Draw The Line,/a/aerosmith/draw+the+line_20004388.html,"Checkmate honey, beat ya at your own damn game...",148,-0.8074,-87.686943
149,Aerosmith,Eat The Rich,/a/aerosmith/eat+the+rich_20004281.html,Well I woke up this morning \r\nOn the wrong ...,149,0.9987,-318.097647
150,Aerosmith,Falling In Love,/a/aerosmith/falling+in+love_10003058.html,"You're so bad, you're so bad, you're so \r\nY...",150,0.9327,-309.590482


In [7]:
print("\n3. Step: Semantic Analysis (TF-IDF + LSA)")

# Creating the TF-IDF Matrix (Stop words: we eliminate unnecessary words in English)
tfidf = TfidfVectorizer(stop_words='english', max_features=4000)
tfidf_matrix = tfidf.fit_transform(df['text'])

print(tfidf_matrix)


3. Step: Semantic Analysis (TF-IDF + LSA)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 401045 stored elements and shape (6813, 4000)>
  Coords	Values
  (0, 2089)	0.08014189637856052
  (0, 2233)	0.3792689714398621
  (0, 2095)	0.5831072710602926
  (0, 1887)	0.1714780013006771
  (0, 1890)	0.4547882034538086
  (0, 1975)	0.11676033059467285
  (0, 159)	0.198746488283843
  (0, 1897)	0.12526241865447826
  (0, 2236)	0.2224422170581528
  (0, 2660)	0.33995710880869606
  (0, 3351)	0.12516187501083195
  (0, 2615)	0.12712716598946586
  (0, 364)	0.07420196467184047
  (0, 1390)	0.04767898605754385
  (1, 2089)	0.021153472823485134
  (1, 159)	0.02622959169082052
  (1, 2449)	0.271265555844839
  (1, 1670)	0.3645386405582536
  (1, 215)	0.050231721993082894
  (1, 3787)	0.13895999406235876
  (1, 669)	0.02678783100614612
  (1, 1763)	0.04083318338503582
  (1, 1892)	0.06780476570883681
  (1, 3489)	0.056961056505062245
  (1, 373)	0.0712232502073733
  :	:
  (6812, 374)	0.05532767527814636
  (681

In [8]:
# Latent Semantic Analysis (LSA) - Dimension Reduction
n_components = 10
lsa = TruncatedSVD(n_components=n_components, random_state=42)
lsa_matrix = lsa.fit_transform(tfidf_matrix)

# Adding LSA results as columns to the main DataFrame
lsa_columns = [f'lsa_dim_{i+1}' for i in range(n_components)]
df_lsa = pd.DataFrame(lsa_matrix, columns=lsa_columns, index=df.index)
df = pd.concat([df, df_lsa], axis=1)

In [9]:
df.head()

,artist,song,link,text,original_index,sentiment_score,readability_score,lsa_dim_1,lsa_dim_2,lsa_dim_3,lsa_dim_4,lsa_dim_5,lsa_dim_6,lsa_dim_7,lsa_dim_8,lsa_dim_9,lsa_dim_10
145,Aerosmith,All Your Love,/a/aerosmith/all+your+love_20004378.html,All your love I miss lovin' \r\nAll your kiss...,145,0.9897,51.873355,0.112774,0.067347,0.036292,0.081858,0.106039,0.015360,-0.088974,-0.083473,-0.013681,-0.025146
146,Aerosmith,Bacon Biscuit Blues,/a/aerosmith/bacon+biscuit+blues_20004481.html,"Put your biscuits in the oven \r\nHoney, put ...",146,-0.8555,-99.192081,0.124590,-0.038656,0.012062,0.041182,0.025831,-0.001987,-0.013887,0.012314,-0.015600,-0.003887
148,Aerosmith,Draw The Line,/a/aerosmith/draw+the+line_20004388.html,"Checkmate honey, beat ya at your own damn game...",148,-0.8074,-87.686943,0.145313,-0.057039,0.062068,-0.034726,-0.038570,-0.021934,0.032375,-0.044483,-0.003167,-0.021141
149,Aerosmith,Eat The Rich,/a/aerosmith/eat+the+rich_20004281.html,Well I woke up this morning \r\nOn the wrong ...,149,0.9987,-318.097647,0.108765,-0.070082,-0.022010,0.011456,-0.007778,-0.002756,-0.031964,-0.005570,-0.000963,-0.027057
150,Aerosmith,Falling In Love,/a/aerosmith/falling+in+love_10003058.html,"You're so bad, you're so bad, you're so \r\nY...",150,0.9327,-309.590482,0.269254,0.052248,-0.052022,0.128778,-0.102611,-0.037403,-0.032616,0.062088,0.028528,-0.067553


In [10]:
print("\n4. Step: Data Standardization")
features_to_scale = ['sentiment_score', 'readability_score'] + lsa_columns

scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[features_to_scale])

# Adding scaled data as new columns (prefixing them with 'scaled_')
scaled_columns = [f'scaled_{col}' for col in features_to_scale]
df_scaled = pd.DataFrame(scaled_features, columns=scaled_columns, index=df.index)
df = pd.concat([df, df_scaled], axis=1)


4. Step: Data Standardization


In [11]:
df.head()

,artist,song,link,text,original_index,sentiment_score,readability_score,lsa_dim_1,lsa_dim_2,lsa_dim_3,lsa_dim_4,lsa_dim_5,lsa_dim_6,lsa_dim_7,lsa_dim_8,lsa_dim_9,lsa_dim_10,scaled_sentiment_score,scaled_readability_score,scaled_lsa_dim_1,scaled_lsa_dim_2,scaled_lsa_dim_3,scaled_lsa_dim_4,scaled_lsa_dim_5,scaled_lsa_dim_6,scaled_lsa_dim_7,scaled_lsa_dim_8,scaled_lsa_dim_9,scaled_lsa_dim_10
145,Aerosmith,All Your Love,/a/aerosmith/all+your+love_20004378.html,All your love I miss lovin' \r\nAll your kiss...,145,0.9897,51.873355,0.112774,0.067347,0.036292,0.081858,0.106039,0.015360,-0.088974,-0.083473,-0.013681,-0.025146,0.720001,0.929217,-0.968480,0.770964,0.427709,1.117986,1.447081,0.160710,-1.175140,-1.217484,-0.228560,-0.340239
146,Aerosmith,Bacon Biscuit Blues,/a/aerosmith/bacon+biscuit+blues_20004481.html,"Put your biscuits in the oven \r\nHoney, put ...",146,-0.8555,-99.192081,0.124590,-0.038656,0.012062,0.041182,0.025831,-0.001987,-0.013887,0.012314,-0.015600,-0.003887,-1.537682,-0.318633,-0.833450,-0.283625,0.159351,0.614453,0.406078,-0.076003,-0.113175,0.208194,-0.257798,-0.012108
148,Aerosmith,Draw The Line,/a/aerosmith/draw+the+line_20004388.html,"Checkmate honey, beat ya at your own damn game...",148,-0.8074,-87.686943,0.145313,-0.057039,0.062068,-0.034726,-0.038570,-0.021934,0.032375,-0.044483,-0.003167,-0.021141,-1.478829,-0.223597,-0.596639,-0.466502,0.713193,-0.325194,-0.429769,-0.348205,0.541104,-0.637161,-0.068305,-0.278418
149,Aerosmith,Eat The Rich,/a/aerosmith/eat+the+rich_20004281.html,Well I woke up this morning \r\nOn the wrong ...,149,0.9987,-318.097647,0.108765,-0.070082,-0.022010,0.011456,-0.007778,-0.002756,-0.031964,-0.005570,-0.000963,-0.027057,0.731013,-2.126864,-1.014299,-0.596264,-0.218014,0.246489,-0.030134,-0.086503,-0.368841,-0.057981,-0.034710,-0.369729
150,Aerosmith,Falling In Love,/a/aerosmith/falling+in+love_10003058.html,"You're so bad, you're so bad, you're so \r\nY...",150,0.9327,-309.590482,0.269254,0.052248,-0.052022,0.128778,-0.102611,-0.037403,-0.032616,0.062088,0.028528,-0.067553,0.650259,-2.056593,0.819732,0.620747,-0.550413,1.698798,-1.260951,-0.559305,-0.378057,0.949030,0.414764,-0.994785


In [12]:
lsa_cols = [f'lsa_dim_{i+1}' for i in range(n_components)]

df['original_index'] = df.index 

df_semantic = df[['original_index', 'sentiment_score', 'readability_score'] + lsa_cols].copy()

df_final = pd.merge(df_pipeline, df_semantic, on='original_index', how='left')

df_final['sentiment_score'] = df_final['sentiment_score_y']
df_final['flesch_kincaid_readability'] = df_final['readability_score']
for i in range(1, n_components + 1):
    df_final[f'lsa_component_{i}'] = df_final[f'lsa_dim_{i}']

final_cols = (
    ['original_index', 'track_id', 'song_name', 'artist', 'genre',
     'chorus_ratio', 'syllable_density',
     'sentiment_score', 'flesch_kincaid_readability'] +
    [f'lsa_component_{i}' for i in range(1, n_components + 1)]
)
df_final = df_final[final_cols]

print(f"Shape: {df_final.shape}")
print(f"Null check:\n{df_final.isnull().sum()}")

df_final.to_csv("final_features.csv", index=False)
print("Saved: final_features.csv")

Shape: (6813, 19)
Null check:
original_index                0
track_id                      0
song_name                     0
artist                        0
genre                         0
chorus_ratio                  0
syllable_density              0
sentiment_score               0
flesch_kincaid_readability    0
lsa_component_1               0
lsa_component_2               0
lsa_component_3               0
lsa_component_4               0
lsa_component_5               0
lsa_component_6               0
lsa_component_7               0
lsa_component_8               0
lsa_component_9               0
lsa_component_10              0
dtype: int64
Saved: final_features.csv
